# 04 Case/Solution Reuse

Notebook ini digunakan untuk tahap keempat sistem Case-Based Reasoning (CBR), yaitu menggunakan kasus lama yang paling mirip sebagai dasar rekomendasi solusi untuk kasus baru.

Input yang digunakan:
- File hasil tahap 3: `stage_03_case_retrieval_output.zip`
- Dataset kasus: `data/processed/cases.csv`
- Model retrieval: `data/processed/tfidf_vectorizer.pkl`
- Matriks TF-IDF: `data/processed/tfidf_matrix.npz`

Metode yang digunakan:
- Retrieval top-k menggunakan TF-IDF dan cosine similarity
- Solution reuse menggunakan:
  1. Amar putusan dari kasus paling mirip
  2. Majority vote untuk label amar
  3. Weighted similarity untuk estimasi lama pidana penjara

Output utama:
- Fungsi `predict_outcome(query, k=5)`
- `data/results/predictions.csv`
- `data/results/prediction_details.csv`
- `stage_04_solution_reuse_output.zip`


## 1. Instalasi Library

Cell ini memasang library yang dibutuhkan untuk membaca data, memuat model TF-IDF, menghitung similarity, dan menyimpan hasil prediksi.


In [ ]:
!pip -q install pandas numpy scikit-learn scipy joblib tqdm openpyxl

## 2. Import Library

Cell ini memanggil library Python yang digunakan pada tahap Case/Solution Reuse.


In [ ]:
import os
import re
import json
import shutil
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

print("Library berhasil dimuat.")

## 3. Menyiapkan Struktur Folder

Cell ini menyiapkan struktur folder project dan memastikan folder output tersedia.


In [ ]:
BASE_DIR = Path("/content/cbr-putusan-penganiayaan")

RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder project siap digunakan.")
print(BASE_DIR)

## 4. Upload dan Ekstrak Output Tahap 3

Cell ini meminta file `stage_03_case_retrieval_output.zip`. File tersebut berisi dataset kasus, model TF-IDF, dan hasil retrieval dari tahap sebelumnya.


In [ ]:
EXPECTED_ZIP_NAME = "stage_03_case_retrieval_output.zip"
zip_path = Path("/content") / EXPECTED_ZIP_NAME

if not zip_path.exists():
    try:
        from google.colab import files
        print("Silakan upload file stage_03_case_retrieval_output.zip")
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError("Tidak ada file yang diupload.")
        zip_path = Path("/content") / list(uploaded.keys())[0]
    except Exception as e:
        raise RuntimeError(
            "File ZIP tahap 3 belum tersedia. Upload file stage_03_case_retrieval_output.zip terlebih dahulu."
        ) from e

print(f"File ZIP digunakan: {zip_path}")

extract_temp = Path("/content/stage_03_extract_temp")
if extract_temp.exists():
    shutil.rmtree(extract_temp)
extract_temp.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_temp)

candidate_data_dirs = list(extract_temp.rglob("data"))

if len(candidate_data_dirs) == 0:
    raise FileNotFoundError("Folder data tidak ditemukan di dalam ZIP tahap 3.")

source_root = candidate_data_dirs[0].parent

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
shutil.copytree(source_root, BASE_DIR)

RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output tahap 3 berhasil diekstrak.")

## 5. Memuat Dataset dan Model Retrieval

Cell ini memuat `cases.csv`, TF-IDF vectorizer, dan matriks TF-IDF yang sudah dibuat pada tahap 3.


In [ ]:
cases_csv_path = PROCESSED_DIR / "cases.csv"
vectorizer_path = PROCESSED_DIR / "tfidf_vectorizer.pkl"
tfidf_matrix_path = PROCESSED_DIR / "tfidf_matrix.npz"

if not cases_csv_path.exists():
    raise FileNotFoundError("cases.csv tidak ditemukan.")
if not vectorizer_path.exists():
    raise FileNotFoundError("tfidf_vectorizer.pkl tidak ditemukan.")
if not tfidf_matrix_path.exists():
    raise FileNotFoundError("tfidf_matrix.npz tidak ditemukan.")

cases_df = pd.read_csv(cases_csv_path)
vectorizer = joblib.load(vectorizer_path)
tfidf_matrix = sparse.load_npz(tfidf_matrix_path)

print("Dataset dan model berhasil dimuat.")
print("Ukuran dataset:", cases_df.shape)
print("Ukuran TF-IDF matrix:", tfidf_matrix.shape)

display(cases_df[["case_id", "nomor_perkara", "pasal", "amar_label"]].head())

## 6. Fungsi Preprocessing dan Retrieval

Cell ini mendefinisikan kembali fungsi preprocessing dan retrieval agar dapat digunakan pada tahap reuse. Fungsi `retrieve()` akan mengembalikan top-k kasus lama yang paling mirip dengan query baru.


In [ ]:
def preprocess_text(text):
    """
    Membersihkan teks agar konsisten dengan tahap retrieval sebelumnya.
    """
    if not isinstance(text, str):
        text = ""

    text = text.lower()

    text = text.replace("kuhpidana", "kuhp")
    text = text.replace("kuh pidana", "kuhp")
    text = text.replace("pidanaidana", "pidana")
    text = text.replace("nopember", "november")
    text = text.replace("pebruari", "februari")

    text = re.sub(r"[^a-zA-Z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()

    return text


def retrieve(query: str, k: int = 5, exclude_case_id: str = None):
    """
    Mengambil top-k kasus paling mirip berdasarkan TF-IDF dan cosine similarity.
    """
    clean_query = preprocess_text(query)
    query_vector = vectorizer.transform([clean_query])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    result_df = cases_df.copy()
    result_df["similarity_score"] = similarities

    if exclude_case_id is not None:
        result_df = result_df[result_df["case_id"] != exclude_case_id]

    result_df = result_df.sort_values(by="similarity_score", ascending=False)

    selected_columns = [
        "case_id",
        "nomor_perkara",
        "tanggal_putusan",
        "terdakwa",
        "pasal",
        "amar_label",
        "similarity_score",
        "ringkasan_fakta",
        "amar_putusan"
    ]

    available_columns = [col for col in selected_columns if col in result_df.columns]

    return result_df[available_columns].head(k).reset_index(drop=True)


## 7. Ekstraksi Solusi dari Amar Putusan

Cell ini mengekstrak solusi dari setiap kasus. Solusi utama yang digunakan adalah `amar_putusan`, sedangkan lama pidana penjara diekstrak untuk membantu prediksi solusi secara numerik.


In [ ]:
number_words = {
    "satu": 1,
    "dua": 2,
    "tiga": 3,
    "empat": 4,
    "lima": 5,
    "enam": 6,
    "tujuh": 7,
    "delapan": 8,
    "sembilan": 9,
    "sepuluh": 10,
    "sebelas": 11,
    "dua belas": 12,
    "tiga belas": 13,
    "empat belas": 14,
    "lima belas": 15,
    "enam belas": 16,
    "tujuh belas": 17,
    "delapan belas": 18,
    "sembilan belas": 19,
    "dua puluh": 20,
    "tiga puluh": 30,
}


def word_to_number(value):
    """
    Mengubah angka dalam bentuk kata Indonesia menjadi integer sederhana.
    """
    if value is None:
        return None

    value = str(value).lower().strip()

    if value.isdigit():
        return int(value)

    value = re.sub(r"[^a-z\\s]", " ", value)
    value = re.sub(r"\\s+", " ", value).strip()

    if value in number_words:
        return number_words[value]

    return None


def extract_sentence_months(amar_text):
    """
    Mengekstrak lama pidana penjara dari amar putusan dalam satuan bulan.
    Contoh:
    - 3 bulan -> 3
    - 1 tahun -> 12
    - 1 tahun 6 bulan -> 18
    """
    if not isinstance(amar_text, str):
        return None

    text = amar_text.lower()
    text = re.sub(r"\\s+", " ", text)

    # Ambil bagian setelah frasa pidana penjara bila tersedia
    match_section = re.search(r"pidana\\s+penjara\\s+selama\\s+(.*?)(?:;|\\.|,| menetapkan| membebankan| menetapkan)", text)
    if match_section:
        section = match_section.group(1)
    else:
        section = text

    total_months = 0
    found = False

    # Pola tahun, misalnya 1 (satu) tahun atau satu tahun
    year_patterns = [
        r"(\\d+)\\s*\\([^)]*\\)\\s*tahun",
        r"([a-z\\s]+?)\\s*tahun"
    ]

    for pattern in year_patterns:
        match = re.search(pattern, section)
        if match:
            value = match.group(1).strip()
            num = word_to_number(value)
            if num is not None:
                total_months += num * 12
                found = True
                break

    # Pola bulan, misalnya 6 (enam) bulan atau enam bulan
    month_patterns = [
        r"(\\d+)\\s*\\([^)]*\\)\\s*bulan",
        r"([a-z\\s]+?)\\s*bulan"
    ]

    for pattern in month_patterns:
        match = re.search(pattern, section)
        if match:
            value = match.group(1).strip()
            # Cegah bagian terlalu panjang dari pola kata
            words = value.split()
            if len(words) > 3:
                value = " ".join(words[-2:])
            num = word_to_number(value)
            if num is not None:
                total_months += num
                found = True
                break

    if found:
        return int(total_months)

    return None


def summarize_solution(amar_text, max_chars=700):
    """
    Membuat ringkasan solusi dari amar putusan.
    """
    if not isinstance(amar_text, str):
        return ""

    text = re.sub(r"\\s+", " ", amar_text).strip()

    # Ambil bagian inti mulai dari 'Menyatakan Terdakwa' jika tersedia
    match = re.search(r"(Menyatakan\\s+Terdakwa.*)", text, flags=re.IGNORECASE)
    if match:
        text = match.group(1)

    return text[:max_chars]


cases_df["sentence_months"] = cases_df["amar_putusan"].apply(extract_sentence_months)
cases_df["solution_text"] = cases_df["amar_putusan"].apply(summarize_solution)

cases_df[["case_id", "nomor_perkara", "amar_label", "sentence_months", "solution_text"]].head(10)


## 8. Membuat Case Solutions Dictionary

Cell ini membuat dictionary `{case_id: solusi_text}` agar solusi setiap kasus mudah dipanggil pada fungsi reuse.


In [ ]:
case_solutions = dict(zip(cases_df["case_id"], cases_df["solution_text"]))

print("Jumlah solusi kasus:", len(case_solutions))

# Contoh solusi
sample_case_id = cases_df.iloc[0]["case_id"]
print("Contoh case:", sample_case_id)
print(case_solutions[sample_case_id][:1000])

## 9. Fungsi Solution Reuse

Cell ini berisi fungsi untuk menggunakan kembali solusi dari kasus lama. Prediksi solusi dibuat berdasarkan top-5 kasus termirip, majority vote pada label amar, dan weighted similarity untuk estimasi lama pidana.


In [ ]:
def weighted_average_sentence(top_k_df):
    """
    Menghitung estimasi lama pidana penjara berdasarkan bobot similarity.
    """
    merged = top_k_df.merge(
        cases_df[["case_id", "sentence_months"]],
        on="case_id",
        how="left"
    )

    valid_rows = merged.dropna(subset=["sentence_months"]).copy()

    if len(valid_rows) == 0:
        return None

    weights = valid_rows["similarity_score"].astype(float).values
    values = valid_rows["sentence_months"].astype(float).values

    # Jika semua similarity nol, gunakan rata-rata biasa
    if weights.sum() == 0:
        return float(np.mean(values))

    return float(np.average(values, weights=weights))


def majority_vote_label(top_k_df):
    """
    Menentukan label amar dengan majority vote.
    Jika seri, pilih label dari kasus dengan similarity tertinggi.
    """
    if "amar_label" not in top_k_df.columns:
        return "tidak_diketahui"

    counts = top_k_df["amar_label"].value_counts()

    if len(counts) == 0:
        return "tidak_diketahui"

    top_count = counts.max()
    candidate_labels = counts[counts == top_count].index.tolist()

    if len(candidate_labels) == 1:
        return candidate_labels[0]

    # Tie-breaker: ambil label dari kasus dengan similarity tertinggi
    for _, row in top_k_df.sort_values("similarity_score", ascending=False).iterrows():
        if row["amar_label"] in candidate_labels:
            return row["amar_label"]

    return candidate_labels[0]


def months_to_sentence_text(months):
    """
    Mengubah jumlah bulan menjadi teks sederhana.
    """
    if months is None or pd.isna(months):
        return "Tidak dapat diestimasi"

    months = int(round(months))

    if months < 12:
        return f"{months} bulan"

    years = months // 12
    remaining_months = months % 12

    if remaining_months == 0:
        return f"{years} tahun"

    return f"{years} tahun {remaining_months} bulan"


def predict_outcome(query: str, k: int = 5):
    """
    Memprediksi solusi kasus baru berdasarkan top-k kasus lama yang paling mirip.

    Output:
    - predicted_label: label solusi berdasarkan majority vote
    - estimated_sentence_months: estimasi lama pidana penjara berbasis weighted similarity
    - recommended_solution: solusi dari kasus paling mirip
    - top_k_cases: daftar kasus termirip
    """
    top_k = retrieve(query, k=k)

    predicted_label = majority_vote_label(top_k)
    estimated_months = weighted_average_sentence(top_k)
    estimated_sentence_text = months_to_sentence_text(estimated_months)

    top_1 = top_k.iloc[0]
    recommended_solution = top_1["amar_putusan"]

    prediction = {
        "predicted_label": predicted_label,
        "estimated_sentence_months": None if estimated_months is None else round(float(estimated_months), 2),
        "estimated_sentence_text": estimated_sentence_text,
        "recommended_solution_from_case_id": top_1["case_id"],
        "recommended_solution_from_nomor_perkara": top_1["nomor_perkara"],
        "recommended_solution": recommended_solution,
        "top_k_cases": top_k
    }

    return prediction


## 10. Uji Prediksi dengan Satu Query Manual

Cell ini menguji fungsi `predict_outcome()` dengan contoh kasus baru. Sistem akan mencari top-5 kasus termirip dan mengambil solusi dari kasus lama.


In [ ]:
query_manual = '''
Terdakwa melakukan penganiayaan kepada korban karena terjadi cekcok.
Terdakwa memukul wajah korban menggunakan tangan beberapa kali sehingga korban mengalami luka dan rasa sakit.
Perbuatan tersebut termasuk penganiayaan sebagaimana Pasal 351 ayat (1) KUHP.
'''

prediction = predict_outcome(query_manual, k=5)

print("PREDICTED LABEL:", prediction["predicted_label"])
print("ESTIMASI PIDANA:", prediction["estimated_sentence_text"])
print("SOLUSI DIAMBIL DARI CASE:", prediction["recommended_solution_from_case_id"])
print("NOMOR PERKARA:", prediction["recommended_solution_from_nomor_perkara"])

print("\nTOP-5 KASUS TERMIRIP:")
display(prediction["top_k_cases"][["case_id", "nomor_perkara", "similarity_score", "amar_label"]])

print("\nREKOMENDASI SOLUSI:")
print(str(prediction["recommended_solution"])[:1500])

## 11. Menyiapkan 5 Contoh Kasus Baru

Cell ini menyiapkan 5 contoh query kasus baru untuk demo manual. Setiap query dibuat berdasarkan variasi fakta umum dalam perkara penganiayaan.


In [ ]:
new_case_queries = [
    {
        "query_id": "new_case_001",
        "query_text": "Terdakwa memukul wajah korban menggunakan tangan setelah terjadi pertengkaran. Korban mengalami luka dan rasa sakit pada wajah. Perbuatan terdakwa termasuk penganiayaan Pasal 351 ayat (1) KUHP."
    },
    {
        "query_id": "new_case_002",
        "query_text": "Terdakwa melempar korban menggunakan batu sehingga korban mengalami luka robek pada bagian bibir. Kejadian terjadi setelah terdakwa mengejar korban dan melakukan kekerasan fisik."
    },
    {
        "query_id": "new_case_003",
        "query_text": "Terdakwa menyerang korban menggunakan benda tajam sehingga korban mengalami luka pada bagian telinga dan leher. Akibat perbuatan tersebut korban terganggu melakukan aktivitas sehari-hari."
    },
    {
        "query_id": "new_case_004",
        "query_text": "Terdakwa melakukan penganiayaan karena merasa tersinggung. Terdakwa menendang dan memukul korban hingga korban mengalami luka memar dan rasa sakit pada tubuh."
    },
    {
        "query_id": "new_case_005",
        "query_text": "Terdakwa melakukan penganiayaan terhadap korban di lokasi parkir setelah terjadi perselisihan. Korban dipukul dan mengalami luka sehingga melaporkan kejadian tersebut kepada pihak kepolisian."
    }
]

print("Jumlah query kasus baru:", len(new_case_queries))
new_case_queries

## 12. Menjalankan Prediksi untuk 5 Kasus Baru

Cell ini menjalankan `predict_outcome()` pada seluruh query demo dan menyimpan hasil prediksi ke tabel.


In [ ]:
prediction_rows = []
detail_rows = []

for item in tqdm(new_case_queries, desc="Memprediksi kasus baru"):
    pred = predict_outcome(item["query_text"], k=5)
    top_k_df = pred["top_k_cases"]

    top_case_ids = top_k_df["case_id"].tolist()
    top_nomor_perkara = top_k_df["nomor_perkara"].tolist()
    top_scores = top_k_df["similarity_score"].round(6).tolist()

    prediction_rows.append({
        "query_id": item["query_id"],
        "query_text": item["query_text"],
        "predicted_label": pred["predicted_label"],
        "estimated_sentence_months": pred["estimated_sentence_months"],
        "estimated_sentence_text": pred["estimated_sentence_text"],
        "recommended_solution_from_case_id": pred["recommended_solution_from_case_id"],
        "recommended_solution_from_nomor_perkara": pred["recommended_solution_from_nomor_perkara"],
        "top_5_case_ids": ", ".join(top_case_ids),
        "top_5_nomor_perkara": ", ".join(top_nomor_perkara),
        "top_5_similarity_scores": ", ".join(map(str, top_scores)),
        "recommended_solution": pred["recommended_solution"]
    })

    for rank, (_, row) in enumerate(top_k_df.iterrows(), start=1):
        detail_rows.append({
            "query_id": item["query_id"],
            "rank": rank,
            "case_id": row["case_id"],
            "nomor_perkara": row["nomor_perkara"],
            "similarity_score": row["similarity_score"],
            "amar_label": row.get("amar_label", ""),
            "amar_putusan": row.get("amar_putusan", "")
        })

predictions_df = pd.DataFrame(prediction_rows)
prediction_details_df = pd.DataFrame(detail_rows)

display(predictions_df[[
    "query_id",
    "predicted_label",
    "estimated_sentence_text",
    "recommended_solution_from_case_id",
    "recommended_solution_from_nomor_perkara"
]])

## 13. Menampilkan Detail Prediksi

Cell ini menampilkan top-5 kasus termirip untuk setiap query baru. Bagian ini berguna untuk menjelaskan proses reuse saat demo.


In [ ]:
prediction_details_df[[
    "query_id",
    "rank",
    "case_id",
    "nomor_perkara",
    "similarity_score",
    "amar_label"
]].head(25)

## 14. Menyimpan Hasil Prediksi

Cell ini menyimpan hasil prediksi ke folder `data/results/`. File `predictions.csv` menjadi output utama tahap Case/Solution Reuse.


In [ ]:
predictions_path = RESULTS_DIR / "predictions.csv"
prediction_details_path = RESULTS_DIR / "prediction_details.csv"
case_solutions_path = PROCESSED_DIR / "case_solutions.json"

predictions_df.to_csv(predictions_path, index=False)
prediction_details_df.to_csv(prediction_details_path, index=False)

with open(case_solutions_path, "w", encoding="utf-8") as f:
    json.dump(case_solutions, f, ensure_ascii=False, indent=2)

print("File hasil tahap 4 berhasil disimpan:")
print("-", predictions_path)
print("-", prediction_details_path)
print("-", case_solutions_path)

## 15. Membuat Ringkasan Tahap 4

Cell ini membuat ringkasan hasil tahap 4, termasuk jumlah kasus baru yang diuji dan metode reuse yang digunakan.


In [ ]:
stage_04_summary = {
    "jumlah_case_base": len(cases_df),
    "jumlah_query_demo": len(new_case_queries),
    "retrieval_method": "TF-IDF + Cosine Similarity",
    "reuse_method_1": "Top-1 recommended solution from most similar case",
    "reuse_method_2": "Majority vote for amar label",
    "reuse_method_3": "Weighted similarity for sentence duration estimation",
    "output_predictions": str(predictions_path),
    "output_prediction_details": str(prediction_details_path),
    "output_case_solutions": str(case_solutions_path)
}

stage_04_summary_df = pd.DataFrame([stage_04_summary])
stage_04_summary_path = PROCESSED_DIR / "stage_04_summary.csv"
stage_04_summary_df.to_csv(stage_04_summary_path, index=False)

stage_04_summary_df

## 16. Membuat ZIP Output Tahap 4

Cell ini membuat ZIP project setelah tahap 4 selesai. ZIP ini dapat digunakan untuk melanjutkan ke tahap 5, yaitu evaluasi model.


In [ ]:
output_zip = Path("/content/stage_04_solution_reuse_output.zip")

if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(
    base_name=str(output_zip).replace(".zip", ""),
    format="zip",
    root_dir=BASE_DIR
)

print(f"ZIP output tahap 4 berhasil dibuat: {output_zip}")

## Kesimpulan Tahap 4

Pada tahap ini, sistem telah menggunakan hasil retrieval untuk mengambil solusi dari kasus lama. Solusi diambil dari amar putusan kasus yang paling mirip, sedangkan label amar dan estimasi lama pidana dihitung berdasarkan top-5 kasus termirip. Hasil prediksi disimpan pada `data/results/predictions.csv` dan akan digunakan untuk evaluasi pada tahap berikutnya.
